In [7]:
import pandas as pd
import spacy
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import RandomOverSampler

# Load your CSV file
data = pd.read_csv("supply_chain_project_trustpilot_advanced_merge.csv")

# Orview
data.head()


,Unnamed: 0,Company,Name,Rating_number_customer,Heading,Comment,Stars,Invitation,Dates
0,0,skatedeluxe,Sandra,2,Jederzeit wieder,"<p class=""typography_body-l__v5JLj typography_...",5,Auf Einladung,5. März 2025
1,1,skatedeluxe,customer,2,Schnelle Lieferung,No comment,5,Auf Einladung,5. März 2025
2,2,skatedeluxe,Dexter,1,Bester Service und top Qualität,"<p class=""typography_body-l__v5JLj typography_...",5,Auf Einladung,4. März 2025
3,3,skatedeluxe,Stephan Lameck,1,Schnelligkeit,"<p class=""typography_body-l__v5JLj typography_...",5,Auf Einladung,4. März 2025
4,4,skatedeluxe,Fritz Brack,2,Super Service,"<p class=""typography_body-l__v5JLj typography_...",5,Auf Einladung,3. März 2025


In [8]:
# Clean up HTML-like text in comments
def extract_comment(text):
    if isinstance(text, str):
        matches = re.findall(r'>([^<]+)<', text)
        if matches:
            return matches[0]
    return text  # If no string or no match, return the text unchanged

# Apply the clean-up
data['Comment'] = data['Comment'].apply(extract_comment)

# Merge headline and comment
data['Text'] = data['Heading'].fillna('') + ' ' + data['Comment'].fillna('')

# Remove unused columns
data = data.drop(['Name', 'Heading', 'Comment'], axis=1)

# Orview
data.head()

,Unnamed: 0,Company,Rating_number_customer,Stars,Invitation,Dates,Text
0,0,skatedeluxe,2,5,Auf Einladung,5. März 2025,"Jederzeit wieder Sehr schnelle Lieferung, gute..."
1,1,skatedeluxe,2,5,Auf Einladung,5. März 2025,Schnelle Lieferung No comment
2,2,skatedeluxe,1,5,Auf Einladung,4. März 2025,Bester Service und top Qualität Der bestellvor...
3,3,skatedeluxe,1,5,Auf Einladung,4. März 2025,Schnelligkeit Ausgefallene Produkte
4,4,skatedeluxe,2,5,Auf Einladung,3. März 2025,"Super Service Super Service, extrem schnelle L..."


In [9]:
# Load the German language model
nlp = spacy.load('de_core_news_sm')

# Tokeniser + Lemmatiser combined
def preprocess_text(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return tokens

# Add tokenised and lemmatised text
data['tokens'] = data['Text'].apply(preprocess_text)

# Only for TF-IDF later, convert back to space-separated text
data['clean_text'] = data['tokens'].apply(lambda x: ' '.join(x))


In [10]:
# Orview
data.head()


,Unnamed: 0,Company,Rating_number_customer,Stars,Invitation,Dates,Text,tokens,clean_text
0,0,skatedeluxe,2,5,Auf Einladung,5. März 2025,"Jederzeit wieder Sehr schnelle Lieferung, gute...","[Jederzeit, schnell, Lieferung, Preis-Leistung...",Jederzeit schnell Lieferung Preis-Leistungsver...
1,1,skatedeluxe,2,5,Auf Einladung,5. März 2025,Schnelle Lieferung No comment,"[schnell, Lieferung, --, comment]",schnell Lieferung -- comment
2,2,skatedeluxe,1,5,Auf Einladung,4. März 2025,Bester Service und top Qualität Der bestellvor...,"[Bester, Service, Top, Qualität, Bestellvorgan...",Bester Service Top Qualität Bestellvorgang unk...
3,3,skatedeluxe,1,5,Auf Einladung,4. März 2025,Schnelligkeit Ausgefallene Produkte,"[Schnelligkeit, Ausgefallene, Produkt]",Schnelligkeit Ausgefallene Produkt
4,4,skatedeluxe,2,5,Auf Einladung,3. März 2025,"Super Service Super Service, extrem schnelle L...","[Super, Service, Super, Service, extrem, schne...",Super Service Super Service extrem schnell Lie...


In [11]:
# Conversion from multiclass to binary classification

# 1 is True for good and 0 is false for bad
# When x is 4 or 5 then is this good and else bad, that do the lambda function
data['Stars_binary'] = data['Stars'].apply(lambda x: 1 if x >= 4 else 0)

# Now you have the binary target variable that you can use in your models
y_binary = data['Stars_binary']

# Separation of characteristics and target variables
X = data['clean_text']
y = y_binary

# Train-Test-Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
# Vektorisierung with TF-IDF
vec = TfidfVectorizer()
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)


In [14]:
# Logistic Regression Modell
model_logreg = LogisticRegression(max_iter=1000, random_state=42)

# Parameter-Grid for Logistic Regression
param_grid_logreg = {
    'C': [0.1, 1, 10],              # Regularisation
    'penalty': ['l1', 'l2'],        # Types of regularisation
    'solver': ['liblinear', 'saga'] # Solver selection (important for L1 regularisation)
}

# Apply GridSearch to logistic regression
grid_search_logreg = GridSearchCV(estimator=model_logreg, param_grid=param_grid_logreg,
                                  scoring='accuracy', cv=5, n_jobs=-1, verbose=1)

# Train on the data
grid_search_logreg.fit(X_train_vec, y_train)

# Best parameters and score
print("Beste Parameter:", grid_search_logreg.best_params_)
print("Beste Accuracy:", grid_search_logreg.best_score_)

# Testing with the test data
best_logreg = grid_search_logreg.best_estimator_
y_pred = best_logreg.predict(X_test_vec)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Beste Parameter: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}
Beste Accuracy: 0.9457890208106363
Test Accuracy: 0.9461371055495104
              precision    recall  f1-score   support

           0       0.93      0.91      0.92      1317
           1       0.95      0.96      0.96      2359

    accuracy                           0.95      3676
   macro avg       0.94      0.94      0.94      3676
weighted avg       0.95      0.95      0.95      3676



In [15]:
# SVM Modell
svm = SVC(random_state=42)

# Parameter grid for SVM (including 'gamma')
param_grid_svm = {
    'C': [0.1, 1, 10],             # Regularisierung
    'kernel': ['linear', 'rbf'],   # Kernel-Auswahl
    'gamma': ['scale', 'auto']     # Gamma-Werte
}

# Apply GridSearch to SVM
grid_search_svm = GridSearchCV(estimator=svm, param_grid=param_grid_svm,
                               scoring='accuracy', cv=5, n_jobs=-1, verbose=1)

# Train on the data
grid_search_svm.fit(X_train_vec, y_train)

# Best parameters and score
print("Beste Parameter:", grid_search_svm.best_params_)
print("Beste Accuracy:", grid_search_svm.best_score_)

# Testing with the test data
best_svm = grid_search_svm.best_estimator_
y_pred = best_svm.predict(X_test_vec)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Beste Parameter: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Beste Accuracy: 0.9542915200762385
Test Accuracy: 0.9542981501632208
              precision    recall  f1-score   support

           0       0.95      0.92      0.94      1317
           1       0.96      0.97      0.96      2359

    accuracy                           0.95      3676
   macro avg       0.95      0.95      0.95      3676
weighted avg       0.95      0.95      0.95      3676



In [17]:
# Grid Search applied to Rondom Forest
# Model
model_rf = RandomForestClassifier(random_state=42)

# Parametergrid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20]
}

# GridSearch
grid_search_rf = GridSearchCV(estimator=model_rf, param_grid=param_grid,
                           scoring='accuracy', cv=5, n_jobs=-1, verbose=1)

# Train on resampled data!
grid_search_rf.fit(X_train_vec, y_train)


# Beste Parameter + Score
print("Beste Parameter:", grid_search_rf.best_params_)
print("Beste Accuracy:", grid_search_rf.best_score_)

best_rf = grid_search_rf.best_estimator_
y_pred = best_rf.predict(X_test_vec)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Beste Parameter: {'max_depth': None, 'n_estimators': 200}
Beste Accuracy: 0.9444288698138215
Test Accuracy: 0.9385201305767138
              precision    recall  f1-score   support

           0       0.93      0.90      0.91      1317
           1       0.95      0.96      0.95      2359

    accuracy                           0.94      3676
   macro avg       0.94      0.93      0.93      3676
weighted avg       0.94      0.94      0.94      3676

